# A3 · Target encoding from scratch: smoothing and out-of-fold encoding

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Implement target encoding with smoothing, in a naive version and in an out-of-fold version, and measure how much the naive version
leaks the target: on a column of **pure noise** it should be worth 0.5 of AUC.

## Dataset

`ds.loans(with_missing=False)`: `occupation` (40 levels, with a real effect on the risk) and `noise`, a
categorical column with 300 levels assigned **at random**, with no link to the target `defaulted`. 75/25 split.

## Educational purpose

Understand why target encoding is the most powerful and the most dangerous method for high-cardinality variables: the mean of the
target of a category computed including the row itself **contains the target of that row**.

## Task

1. Write `te_estimate(cat, y, m)`: for each category compute `(sum_y + m · global_mean) / (count + m)` (the *smoothing* pulls the rare categories toward the global mean). Return `(mapping, overall)`, with `mapping` a dictionary category → value.
2. Write `te_apply(cat, mapping, overall)`: it encodes an array of categories; categories not present in the mapping get `overall`.
3. Write `te_oof(cat, y, m, k=5, seed=0)`: out-of-fold encoding. With `KFold(k, shuffle=True, random_state=seed)`, every row of a fold is encoded with the mapping computed on the **other** folds.
4. With `m = 5` build for `occupation` and for `noise`: the naive encoding on the training set (`te_estimate` and `te_apply` on the same training set), the out-of-fold one and the one of the test set (mapping of the whole training set). Save in `table` (rows `occupation`, `noise`; columns `naive_train_auc`, `oof_train_auc`, `test_auc`) the AUC with which **each encoded column, alone**, predicts the target (`roc_auc_score(y, encoding)`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split

df = ds.loans(with_missing=False)
y = df["defaulted"].to_numpy()
occ = df["occupation"].to_numpy()
noise = np.array([f"r{i}" for i in np.random.default_rng(1).integers(0, 300, len(df))])   # no link to y
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.25, random_state=0)
y_tr, y_te = y[idx_tr], y[idx_te]

## Your work

In [ ]:
def te_estimate(cat, y, m):
    ...
    return mapping, overall


def te_apply(cat, mapping, overall):
    ...


def te_oof(cat, y, m, k=5, seed=0):
    ...

In [ ]:
m = 5
rows = {}
for name, col in [("occupation", occ), ("noise", noise)]:
    c_tr, c_te = col[idx_tr], col[idx_te]
    ...
    rows[name] = {"naive_train_auc": ..., "oof_train_auc": ..., "test_auc": ...}
table = pd.DataFrame(rows).T
table.round(3)

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='With pandas: `g = pd.DataFrame({"c": cat, "y": y}).groupby("c")["y"].agg(["sum", "count"])` and then `((g["sum"] + m * overall) / (g["count"] + m)).to_dict()`. To apply: `pd.Series(cat).map(mapping).fillna(overall).to_numpy()`. For the out-of-fold: `for tr, va in KFold(...).split(cat): mapping, g = te_estimate(cat[tr], y[tr], m); out[va] = te_apply(cat[va], mapping, g)`.'):
    mapping, glob = te_estimate(np.array(["a", "a", "b"]), np.array([1, 0, 1]), m=1)
    check(close(glob, 2 / 3) and close(mapping["a"], (1 + 2 / 3) / 3) and close(mapping["b"], (1 + 2 / 3) / 2),
          "te_estimate is correct on a hand example", "with m=1: a -> (1 + 2/3)/3, b -> (1 + 2/3)/2")
    check(close(te_apply(np.array(["a", "z"]), mapping, glob), [mapping["a"], glob]),
          "te_apply uses the global mean for unknown categories", "a category not in the mapping gets `overall`")
    c_toy = np.array(list("aabbabab") * 5)
    y_toy = np.tile([1, 0, 0, 1, 1, 1, 0, 0], 5)
    enc = te_oof(c_toy, y_toy, m=2, k=5, seed=0)
    check(enc.shape == (40,) and not np.isnan(enc).any(), "te_oof returns one value per row", "one value for every row")
    check(not np.allclose(enc, te_apply(c_toy, *te_estimate(c_toy, y_toy, 2))),
          "the out-of-fold encoding does NOT match the naive one", "every row must be encoded without itself")
    check(table.loc["noise", "naive_train_auc"] > 0.75,
          "on pure noise the naive encoding seems to predict well (high training AUC): it is leakage",
          "naive_train_auc of the noise should be high: are you encoding the training set with the mapping computed on the training set?")
    check(abs(table.loc["noise", "oof_train_auc"] - 0.5) < 0.08,
          "on pure noise the out-of-fold encoding gives a training AUC close to 0.5",
          "with the out-of-fold encoding the noise should have an AUC of about 0.5")
    check(table.loc["noise", "test_auc"] < 0.65, "on the test set the noise predicts nothing",
          "the test AUC of the noise must stay close to 0.5")
    check(table.loc["occupation", "naive_train_auc"] - table.loc["occupation", "oof_train_auc"] > 0.06,
          "even on a real variable the naive encoding is inflated", "the naive training AUC must exceed the out-of-fold one by at least 0.06")
done()

---
The commented solution is in `solutions/A_from_scratch/A3_target_encoding_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.